# Clase 02. Análisis exploratorio de datos

**INF-396 Introducción a la Ciencia de Datos**
Universidad Técnica Federico Santa María, Departamento de Informática

Repaso de Pandas y primera parte del análisis exploratorio de datos (unidad 2 del programa).

## El concepto de hoy: ¿de quién habla este número?

Ustedes ya trabajaron con Pandas, así que la mecánica de hoy va a paso de repaso. Lo
que esta clase agrega es una pregunta que va a reaparecer durante todo el curso: cada
vez que una cifra resuma datos (un promedio, un porcentaje, un total), pregúntense
**de quién está hablando ese número**.

Lo que sigue son variaciones de esa pregunta: un promedio puede describir una
situación en la que pocos están; contar filas de una encuesta habla de la muestra y no
de la ciudad; y hay personas de las que la tabla no dice nada, y esa ausencia también
es información.

Las secciones de este notebook se organizan por preguntas sobre los datos; las
operaciones de Pandas aparecen donde se necesitan. La pregunta que las une viene
de la ciudad: **¿cómo se mueve Santiago?**

## Dos definiciones del marco conceptual

En las slides de la clase se desarrollan con detalle; aquí quedan las definiciones
de referencia (James et al., 2023, cap. 2).

**Modelo**: una representación simplificada del proceso que genera los datos.
Formalmente se postula $Y = f(X) + \varepsilon$, donde $f$ es la relación
sistemática que se quiere estimar y $\varepsilon$ el error irreducible. Construir
un modelo implica decidir qué variables entran, qué se omite y qué se considera
éxito; como vimos en la clase 1, esas decisiones incorporan las opiniones de quien
modela.

**Inferencia y predicción**: al estimar $f$ puede buscarse comprender la relación
entre las variables (inferencia: qué se asocia con qué, con qué magnitud y signo) o
estimar $Y$ para casos nuevos (predicción, donde $f$ puede tratarse como caja negra
si acierta). Las unidades 2 a 5 del curso son descriptivas e inferenciales; las
unidades 6 a 9, predictivas.

## Los datos: la Encuesta Origen Destino de Santiago

La **EOD 2012** es la encuesta de movilidad del Gran Santiago, levantada para
[SECTRA](https://www.sectra.gob.cl/biblioteca/detalle1.asp?mfn=3253) por la
Universidad Alberto Hurtado: 18.264 hogares y 60.054 personas, con el registro de
todos los viajes que cada persona hizo durante un día asignado. Se usa en la
planificación de transporte de la ciudad.

Es una base **relacional**: varias tablas que se conectan por identificadores.

| Tabla | Una fila es... | Se conecta por |
|-------|----------------|----------------|
| `Hogares.csv` | un hogar encuestado | `Hogar` |
| `personas.csv` | una persona de un hogar | `Hogar`, `Persona` |
| `viajes.csv` | un viaje de una persona | `Hogar`, `Persona`, `Viaje` |
| `tablas_parametros/` | el significado de cada código | según la tabla |

El mismo diseño aparece en la Casen, el Censo o la ENUSC: los datos en tablas
normalizadas y los significados en tablas de códigos.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 25)

RUTA = "datos/eod_stgo/"

## 1. ¿A quiénes les preguntaron?

Una encuesta responde por su muestra, así que lo primero es saber quién está en ella.
Partimos por la tabla de personas. Repaso breve: `read_csv` con el separador correcto
(estos archivos usan punto y coma, y coma decimal), y las tres miradas iniciales:
`shape`, `head`, `info`.

In [ ]:
personas = pd.read_csv(RUTA + "personas.csv", sep=";", decimal=",", low_memory=False)
hogares = pd.read_csv(RUTA + "Hogares.csv", sep=";", decimal=",", low_memory=False)

print(f"Personas encuestadas: {len(personas):,}")
print(f"Hogares encuestados:  {len(hogares):,}")

In [ ]:
personas.head(3)

In [ ]:
personas.info()

Dos observaciones antes de seguir.

Primero, `Persona` no es un identificador único por sí solo: se repite entre hogares.
La llave real es el par `(Hogar, Persona)`. Usar una llave incompleta es un error
frecuente y difícil de detectar, porque el merge igual corre.

Segundo, no hay una columna de edad, pero sí el año de nacimiento. La encuesta se
terminó de levantar en 2013, así que **derivamos** la edad como una columna nueva.

### Una nota sobre el DataFrame

Ya trabajan con DataFrames; dos datos que quizás no conocen. El concepto viene del
lenguaje S, donde los objetos `data.frame` aparecen en *Statistical Models in S*
(Chambers y Hastie, 1992); Wes McKinney lo llevó a Python al crear pandas
(McKinney, 2010). Su rendimiento tiene una explicación concreta: cada columna es un
arreglo tipado y contiguo en memoria, como un arreglo de C: todos los valores son del
mismo tipo y se almacenan crudos, uno al lado del otro, a diferencia de una lista de
Python, que guarda punteros a objetos repartidos por la memoria. Por eso una operación
como `.sum()` se ejecuta
como una sola llamada a código compilado que recorre la columna completa (el tipo se
verifica una vez, y como los datos están contiguos, la memoria caché del procesador trabaja a favor). Un ciclo de
Python, en cambio, pasa por el intérprete en cada elemento: verificar el tipo,
desempaquetar el objeto, operar y volver a empaquetar. A eso se refiere el término
**vectorización**. Además, el índice alinea los datos entre tablas, y `merge` y
`groupby` implementan el álgebra relacional en memoria. La misma interfaz la
replican Spark, Polars y Dask.

La diferencia se puede medir con la columna de edades:

In [ ]:
import time

col = personas["Edad"]

inicio = time.perf_counter()
total = 0
for valor in col:
    total += valor
t_ciclo = time.perf_counter() - inicio

inicio = time.perf_counter()
total_vec = col.sum()
t_vectorizada = time.perf_counter() - inicio

print(f"Ciclo de Python: {t_ciclo * 1000:.2f} ms")
print(f"Vectorizada:     {t_vectorizada * 1000:.3f} ms")
print(f"La versión vectorizada es {t_ciclo / t_vectorizada:.0f} veces más rápida")

In [ ]:
personas["Edad"] = 2013 - personas["AnoNac"]
personas["Edad"].describe().round(1)

### Los tipos de datos de la encuesta

Antes de resumir una variable hay que saber de qué tipo es, porque el tipo determina
qué operaciones tienen sentido:

| Tipo | Qué es | Ejemplos en la EOD | Resúmenes con sentido |
|------|--------|--------------------|------------------------|
| Categórico nominal | categorías sin orden | `Sexo`, `Comuna`, `Proposito` | conteos y proporciones |
| Categórico ordinal | categorías con orden | `Estudios` (nivel educacional) | conteos, mediana de la categoría |
| Numérico discreto | conteos enteros | número de viajes por persona | media, mediana, percentiles |
| Numérico continuo | mediciones | `IngresoHogar`, `TiempoViaje` | media, mediana, percentiles, dispersión |

Un punto que induce a error: **el `dtype` de Pandas no es el tipo estadístico**. En
`personas.info()` la columna `Sexo` aparece como `int64`, pero no es una cantidad:
es una categoría codificada con un número. Pandas no distingue esa diferencia y
calcula lo que se le pida:

In [ ]:
personas["Sexo"].mean().round(3)

El resultado (1,53) es aritméticamente correcto y estadísticamente absurdo: el
promedio de una variable nominal no representa nada. Con `Edad` la misma operación
sí es interpretable, porque la variable es numérica. Distinguir el tipo de cada
columna es un paso del análisis, no una formalidad.

## 2. ¿Quiénes son? El primer merge

Miremos la composición por sexo:

In [ ]:
personas["Sexo"].value_counts()

El conteo es correcto pero ilegible: ¿qué significa ser `1` o ser `2`? La encuesta
guarda **códigos**, y los significados viven en otra tabla. Este es el diseño
estándar de una base de datos, y la operación que conecta ambas partes es `merge`.

In [ ]:
sexo = pd.read_csv(RUTA + "tablas_parametros/Sexo.csv", sep=";")
sexo

In [ ]:
personas = personas.merge(sexo, left_on="Sexo", right_on="Id", how="left",
                          suffixes=("_codigo", ""))
personas["Sexo"].value_counts(normalize=True).round(3)

Después de todo merge: contar cuántas filas quedaron sin pareja. Un código que no está
en la tabla de parámetros produce `NaN` sin avisar.

In [ ]:
print(f"Personas sin sexo asignado tras el merge: {personas['Sexo'].isna().sum()}")

### Las variantes del merge

`merge` tiene tres decisiones y conviene tomarlas de forma explícita.

**1. Por qué columna unir.** `on="columna"` cuando se llama igual en ambas tablas;
`left_on` y `right_on` cuando los nombres difieren (recién usamos `Sexo` contra `Id`).

**2. Qué hacer con las filas sin pareja.** Es el parámetro `how`, y cambia qué filas
sobreviven:

| `how` | Qué conserva |
|-------|--------------|
| `"inner"` | solo las llaves presentes en **ambas** tablas (es el valor por defecto) |
| `"left"` | **todas** las filas de la tabla izquierda; las sin pareja quedan con `NaN` |
| `"right"` | todas las filas de la derecha; las sin pareja quedan con `NaN` |
| `"outer"` | todas las llaves de ambas tablas |

**3. Qué cardinalidad esperar.** Si la llave se repite en una de las tablas, las filas
de la otra se **duplican** para calzar. Al unir personas con viajes, cada persona
aparecerá una vez por cada viaje que hizo: eso es lo esperado en una relación
uno a muchos, pero hay que saber que va a pasar.

Veámoslo con datos. Unamos personas con viajes de las dos maneras y comparemos:

In [ ]:
viajes = pd.read_csv(RUTA + "viajes.csv", sep=";", decimal=",", low_memory=False)

pv_inner = personas.merge(viajes, on=["Hogar", "Persona"], how="inner")
pv_left = personas.merge(viajes, on=["Hogar", "Persona"], how="left")

print(f"personas:            {len(personas):>7,} filas")
print(f"viajes:              {len(viajes):>7,} filas")
print(f"merge how='inner':   {len(pv_inner):>7,} filas")
print(f"merge how='left':    {len(pv_left):>7,} filas")

Lectura del resultado:

- El `inner` tiene una fila por cada viaje con persona conocida. Las personas se
  duplicaron (una fila por viaje): la relación es uno a muchos.
- El `left` agrega además una fila por cada persona **sin viajes**, con las columnas
  del viaje en `NaN`. La diferencia entre ambos conteos es exactamente la cantidad de
  personas que no viajaron.

Y aquí vuelve la pregunta de la clase: **un `inner` hace desaparecer en silencio a
quienes no viajaron**. Si después calculamos "viajes promedio por persona" sobre ese
resultado, el número habla solo de quienes viajan. La elección de `how` no es un
detalle técnico: define de quién hablan los números que vienen después.

## 3. Qué es el análisis exploratorio de datos

Lo que estamos haciendo tiene nombre. El **análisis exploratorio de datos** (EDA,
por *exploratory data analysis*) es la etapa en que se examinan los datos antes de
modelarlos. El término lo introduce John Tukey en 1977, en el libro del mismo
nombre, como contrapeso a una estadística dedicada solo a confirmar hipótesis ya
formuladas: antes de confirmar nada, hay que mirar qué contienen los datos.

En la práctica, el EDA responde cuatro tipos de pregunta:

1. **La forma**: cómo se distribuye cada variable (centro, dispersión, asimetría).
2. **Lo típico y lo extremo**: qué valores son frecuentes y cuáles anómalos.
3. **Lo que falta**: qué datos están ausentes y si esa ausencia sigue un patrón.
4. **Las relaciones**: qué variables se asocian con qué otras.

Se distingue del análisis **confirmatorio** (los tests de hipótesis de la unidad 5)
en el propósito: el EDA genera hipótesis, el confirmatorio las pone a prueba. Una
hipótesis sugerida por los datos no puede confirmarse con esos mismos datos.

Las secciones que siguen recorren esas cuatro preguntas sobre la EOD.

## 4. ¿Cuánto gana un hogar típico?

La tabla de hogares trae el ingreso mensual. Esta es la pregunta por **la forma** de
una distribución y por el número que la resume.

In [ ]:
hogares["IngresoHogar"].describe().round(0)

In [ ]:
media = hogares["IngresoHogar"].mean()
mediana = hogares["IngresoHogar"].median()

print(f"Ingreso medio:   ${media:>12,.0f}")
print(f"Ingreso mediano: ${mediana:>12,.0f}")
print(f"La media es un {media / mediana - 1:.0%} más alta que la mediana")

La media queda por encima de la mediana porque los ingresos son **asimétricos**: la
mayoría de los hogares gana montos moderados y unos pocos ganan mucho más, y esos
pocos desplazan el promedio. Decir "el hogar promedio gana $688 mil" describe una
situación en la que la mayoría de los hogares no está.

Cuando escuchen "el sueldo promedio en Chile", pregunten dónde quedó la mediana.

Los **percentiles** describen la distribución completa:

In [ ]:
hogares["IngresoHogar"].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).round(0)

Un histograma muestra la forma de la asimetría. La próxima clase está dedicada a los
métodos gráficos; por ahora, la versión mínima:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(hogares["IngresoHogar"] / 1e6, bins=60, color="#4C72B0")
ax.axvline(mediana / 1e6, color="#55A868", linewidth=2, label=f"Mediana: ${mediana/1e6:.2f} M")
ax.axvline(media / 1e6, color="#DD8452", linewidth=2, label=f"Media: ${media/1e6:.2f} M")
ax.set_xlim(0, 4)
ax.set_xlabel("Ingreso mensual del hogar (millones de pesos de 2012)")
ax.set_ylabel("Hogares")
ax.set_title("La cola larga de los ingresos desplaza la media")
ax.legend()
plt.show()

## 5. ¿Cuántos viajes hace una persona en un día?

Ya sabemos, por la comparación de merges, que las personas sin viajes no están en la
tabla de viajes. Para contar viajes por persona sin perderlas: agrupar los viajes por
la llave compuesta, y luego alinear ese conteo contra la tabla completa de personas.

In [ ]:
viajes_por_persona = viajes.groupby(["Hogar", "Persona"]).size()

conteo = personas.set_index(["Hogar", "Persona"]).assign(
    n_viajes=viajes_por_persona
)["n_viajes"].fillna(0)

print(f"Promedio de viajes por persona: {conteo.mean():.2f}")
print(f"Mediana de viajes por persona:  {conteo.median():.0f}")
print(f"Personas que no viajaron:       {(conteo == 0).mean():.1%}")

Aquí la media (1,9) quedó **por debajo** de la mediana (2). La asimetría no siempre
empuja hacia el mismo lado: un 23% de las personas no viajó ese día, y ese bloque de
ceros baja el promedio. Las medidas de resumen se interpretan mirando la
distribución, no de memoria.

In [ ]:
conteo.value_counts().sort_index().head(10)

## 6. ¿A quién representa la encuesta? Los factores de expansión

Se encuestó a 60 mil personas; el Gran Santiago tiene millones. ¿Cómo pasa una
encuesta de la muestra a la ciudad?

Mediante el **factor de expansión**: cada persona encuestada representa a un número
determinado de personas parecidas a ella (por zona, edad y perfil). Ese número viene
calculado en `Factor_LaboralNormal` para un día laboral de temporada normal, y la
suma de los factores estima el total de población representada.

In [ ]:
factor = personas["Factor_LaboralNormal"]
print(f"Personas en la muestra con factor laboral: {factor.notna().sum():,}")
print(f"Población que representan:                 {factor.sum():,.0f}")

¿Cambia las respuestas? Comparemos la proporción de mujeres en la muestra contra la
proporción **ponderada**, que es la estimación para la ciudad:

In [ ]:
cruda = personas["Sexo"].value_counts(normalize=True)

ponderada = (
    personas.dropna(subset=["Factor_LaboralNormal"])
    .groupby("Sexo")["Factor_LaboralNormal"].sum()
)
ponderada = ponderada / ponderada.sum()

pd.DataFrame({"muestra": cruda, "ciudad (ponderada)": ponderada}).round(3)

En resumen: **contar filas responde preguntas sobre la muestra; sumar factores
responde preguntas sobre la ciudad**. Las cifras oficiales de esta encuesta (y de la
Casen y la ENUSC) se calculan con factores. En la tabla de viajes el factor del viaje
está en `FactorLaboralNormal`; los viajes de fin de semana traen ese factor vacío
porque pertenecen a otra expansión (`FactorSabadoNormal`, `FactorDomingoNormal`).

## 7. ¿Cuánto dura un viaje?

`TiempoViaje` está en minutos.

In [ ]:
duracion = viajes["TiempoViaje"].dropna()
duracion.describe().round(1)

La mediana dice que el viaje típico dura media hora. El máximo registra más de 1.300
minutos: **un viaje de 22 horas dentro de Santiago**. ¿Error de digitación? ¿Un turno
nocturno mal anotado?

Los valores extremos no se eliminan por incómodos: se investigan, y la decisión que
se tome (mantener, corregir, excluir) se declara en el análisis. Primero, qué tan
raros son:

In [ ]:
print(duracion.quantile([0.90, 0.95, 0.99, 0.999]).round(0))
print(f"\nViajes de más de 3 horas: {(duracion > 180).sum():,} "
      f"({(duracion > 180).mean():.2%} del total)")

## 8. ¿En qué se mueve Santiago?

El modo de transporte para difusión está en `ViajesDifusion.csv`, codificado, con su
tabla de parámetros: dos merges y un groupby ponderado.

In [ ]:
modo_viaje = pd.read_csv(RUTA + "ViajesDifusion.csv", sep=";")
modo_nombres = pd.read_csv(RUTA + "tablas_parametros/ModoDifusion.csv", sep=";")

viajes_modo = (
    viajes.merge(modo_viaje, on="Viaje", how="left")
    .merge(modo_nombres, left_on="ModoDifusion", right_on="ID", how="left",
           suffixes=("_codigo", ""))
)
print(f"Viajes sin modo asignado: {viajes_modo['ModoDifusion'].isna().sum()}")

In [ ]:
reparto = (
    viajes_modo.dropna(subset=["FactorLaboralNormal"])
    .groupby("ModoDifusion")["FactorLaboralNormal"].sum()
    .sort_values(ascending=False)
)
(reparto / reparto.sum()).round(3)

Esta es la **partición modal** del Gran Santiago en día laboral, una cifra de
referencia en la planificación de transporte, calculada aquí desde los microdatos:
la caminata concentra un tercio de los viajes.

### ¿Se mueve igual toda la ciudad?

El total esconde diferencias entre comunas. Para verlas: agregamos la comuna del
hogar a cada viaje (otro merge), simplificamos los modos a cuatro categorías, y
normalizamos por comuna para comparar proporciones.

In [ ]:
def agrupar_modo(modo):
    if modo == "Caminata":
        return "Caminata"
    if isinstance(modo, str) and modo.startswith("Bip!"):
        return "Transporte público"
    if modo == "Auto":
        return "Auto"
    return "Otros"

viajes_modo = viajes_modo.merge(hogares[["Hogar", "Comuna"]], on="Hogar", how="left")
viajes_modo["Modo4"] = viajes_modo["ModoDifusion"].map(agrupar_modo)

modo_comuna = viajes_modo.pivot_table(
    index="Comuna", columns="Modo4", values="FactorLaboralNormal",
    aggfunc="sum", fill_value=0,
)
modo_comuna = modo_comuna.div(modo_comuna.sum(axis=1), axis=0)

# Las cinco comunas con mayor y menor uso de transporte público
orden_tp = modo_comuna.sort_values("Transporte público")
pd.concat([orden_tp.head(5), orden_tp.tail(5)]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
orden_tp[["Caminata", "Transporte público", "Auto", "Otros"]].plot(
    kind="barh", stacked=True, ax=ax,
    color=["#4C72B0", "#DD8452", "#55A868", "#C9CDD3"], width=0.85,
)
ax.set_xlabel("Proporción de los viajes (ponderada)")
ax.set_ylabel("")
ax.set_title("Reparto modal por comuna, ordenado por uso de transporte público")
ax.legend(loc="lower right", framealpha=0.9)
plt.show()

Las comunas con menos uso de transporte público son de dos tipos muy distintos, y
la siguiente pregunta ayuda a separarlos.

### ¿El ingreso explica el uso de transporte público?

Calculamos el ingreso medio ponderado de cada comuna y lo cruzamos con la proporción
de viajes en transporte público.

In [ ]:
ingreso_comuna = (
    hogares.assign(ingreso_pond=hogares["IngresoHogar"] * hogares["Factor"])
    .groupby("Comuna")[["ingreso_pond", "Factor"]].sum()
)
ingreso_comuna = ingreso_comuna["ingreso_pond"] / ingreso_comuna["Factor"]

comunas = pd.DataFrame({
    "ingreso": ingreso_comuna,
    "transporte_publico": modo_comuna["Transporte público"],
}).dropna()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(comunas["ingreso"] / 1e6, comunas["transporte_publico"],
           color="#4C72B0", s=45)
destacadas = ["VITACURA", "LO BARNECHEA", "LA PINTANA", "CONCHALI", "SANTIAGO",
              "BUIN", "COLINA", "MELIPILLA", "PROVIDENCIA", "PUENTE ALTO"]
for nombre in destacadas:
    if nombre in comunas.index:
        fila = comunas.loc[nombre]
        ax.annotate(nombre.title(), (fila["ingreso"] / 1e6, fila["transporte_publico"]),
                    fontsize=8.5, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("Ingreso medio del hogar (millones de pesos, ponderado)")
ax.set_ylabel("Proporción de viajes en transporte público")
ax.set_title("Ingreso y uso de transporte público, por comuna")
plt.show()

### El coeficiente de correlación

Para cuantificar la asociación entre dos variables numéricas se usa el **coeficiente
de correlación de Pearson**, que mide la asociación **lineal** entre ambas y va de
-1 (relación lineal decreciente perfecta) a 1 (creciente perfecta), con 0 indicando
ausencia de relación lineal.

In [ ]:
comunas["ingreso"].corr(comunas["transporte_publico"]).round(2)

### Pearson y Spearman: qué mide cada uno

El coeficiente que calculamos recién es el de **Pearson**, que mide la asociación
**lineal**: qué tan bien se ajustan los puntos a una recta. Es la covarianza
estandarizada,

$$r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i (x_i - \bar{x})^2}\; \sqrt{\sum_i (y_i - \bar{y})^2}}$$

Cada observación aporta el producto de sus desviaciones respecto de la media. De esa
construcción se desprenden sus tres propiedades: un valor extremo entra multiplicando
y puede dominar el resultado; una relación curva produce un coeficiente bajo aunque
la asociación sea fuerte; y transformar los datos cambia su valor.

El coeficiente de **Spearman** es el mismo Pearson, pero calculado sobre los
**rangos**: la posición de cada valor al ordenar la variable. Sin empates equivale a

$$\rho = 1 - \frac{6 \sum_i d_i^2}{n(n^2 - 1)}$$

donde $d_i$ es la diferencia entre los rangos de la observación $i$ en ambas
variables. Al trabajar con posiciones en vez de magnitudes, mide cuán consistente es
que al crecer una variable crezca la otra, **sin importar la forma de la relación**.
Primero, veamos qué le hace el paso a rangos a un valor extremo:

In [ ]:
valores = pd.Series([5, 12, 8, 200])
pd.DataFrame({"valor": valores, "rango": valores.rank().astype(int)})

El 200, por extremo que sea, queda reducido a "el cuarto de cuatro": su magnitud
desaparece y solo sobrevive su posición.

Con dos variables, el cálculo completo es: ordenar cada variable por separado,
asignar los rangos, restar los rangos de cada caso ($d_i$) y aplicar la fórmula.
Con cinco viajes reales de la EOD:

In [ ]:
distancias = pd.read_csv(RUTA + "DistanciaViaje.csv", sep=";", decimal=",")
par = viajes.merge(distancias, on="Viaje", how="left")[["DistEuclidiana", "TiempoViaje"]].dropna()
par = par[(par["DistEuclidiana"] > 0) & (par["TiempoViaje"] > 0)]

cinco = par.sample(5, random_state=1)
ejemplo = pd.DataFrame({
    "distancia_km": (cinco["DistEuclidiana"] / 1000).round(1),
    "duracion_min": cinco["TiempoViaje"].astype(int),
})
ejemplo["rango_dist"] = ejemplo["distancia_km"].rank().astype(int)
ejemplo["rango_dur"] = ejemplo["duracion_min"].rank().astype(int)
ejemplo["d"] = ejemplo["rango_dist"] - ejemplo["rango_dur"]
ejemplo["d2"] = ejemplo["d"] ** 2
ejemplo.sort_values("distancia_km")

In [ ]:
n = len(ejemplo)
rho_manual = 1 - 6 * ejemplo["d2"].sum() / (n * (n**2 - 1))
rho_pandas = ejemplo["distancia_km"].corr(ejemplo["duracion_min"], method="spearman")

print(f"rho con la fórmula: {rho_manual:.2f}")
print(f"rho según pandas:   {rho_pandas:.2f}")

Los rangos de este ejemplo casi coinciden (solo los dos viajes cortos intercambian
posiciones), así que $\sum d_i^2 = 2$ y $\rho = 0{,}90$: un orden casi
perfectamente consistente. La fórmula manual y `pandas` entregan lo mismo. De ahí las propiedades de Spearman: es
robusto a valores extremos, no cambia ante transformaciones monótonas (aplicar
logaritmo no altera el orden), y como solo necesita orden, **es válido para
variables ordinales** como el nivel educacional, donde Pearson no tiene sentido.

Comparemos ambos coeficientes en un par de variables que deberían estar
relacionadas: la distancia de un viaje y su duración. Nos quedamos con los viajes de
distancia y duración positivas, que además necesitaremos para el logaritmo.

In [ ]:
import numpy as np

print(f"Viajes considerados: {len(par):,}")
print(f"Pearson:  {par['DistEuclidiana'].corr(par['TiempoViaje']):.2f}")
print(f"Spearman: {par['DistEuclidiana'].corr(par['TiempoViaje'], method='spearman'):.2f}")

La brecha entre 0,43 y 0,76 es grande, y **esa brecha ya es información**: dice que
la relación es claramente creciente (Spearman alto) pero no lineal o con extremos
que distorsionan (Pearson bajo). Podemos verlo pasando una muestra de viajes a
rangos:

In [ ]:
muestra = par.sample(60, random_state=3)
rangos = muestra.rank()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(muestra["DistEuclidiana"] / 1000, muestra["TiempoViaje"], s=25, color="#4C72B0")
ax1.set_xlabel("Distancia (km)")
ax1.set_ylabel("Duración (minutos)")
ax1.set_title("Escala original: curva, con extremos")

ax2.scatter(rangos["DistEuclidiana"], rangos["TiempoViaje"], s=25, color="#DD8452")
ax2.set_xlabel("Rango de la distancia")
ax2.set_ylabel("Rango de la duración")
ax2.set_title("En rangos: el orden es casi lineal")
plt.show()

Y la verificación final: si la brecha se debe a la curvatura, una transformación
que enderece la curva debería acercar Pearson a Spearman. El logaritmo, al comprimir
los valores grandes, hace exactamente eso:

In [ ]:
r_log = np.log(par["DistEuclidiana"]).corr(np.log(par["TiempoViaje"]))
print(f"Pearson sobre log(distancia) y log(duración): {r_log:.2f}")

Pearson sube de 0,43 a 0,73, casi el valor de Spearman. El logaritmo enderezó la
curva y Pearson recuperó la asociación que Spearman veía desde el principio, porque
Spearman es invariante a esa transformación. La curvatura tiene una explicación de
transporte: los viajes largos usan modos más rápidos (metro, autopistas), así que la
duración crece menos que proporcionalmente con la distancia.

**Cuándo usar cada uno.** Pearson cuando ambas variables son numéricas, el gráfico
se ve aproximadamente recto y no hay extremos dominantes; además su cuadrado
anticipa el R² de la regresión lineal (unidad 7). Spearman con distribuciones
asimétricas o con extremos, relaciones curvas pero crecientes, o variables
ordinales. En la práctica del EDA: calcular ambos, y si difieren mucho, mirar el
gráfico antes de reportar cualquiera.

### Cuando la relación no es monótona, ambos coeficientes fallan

Queda un caso importante: relaciones fuertes que suben y luego bajan. En la EOD, los
viajes por persona según la edad tienen esa forma. Usamos el conteo de viajes que
calculamos antes (con los ceros incluidos):

In [ ]:
edad_viajes = pd.DataFrame({
    "Edad": personas.set_index(["Hogar", "Persona"])["Edad"],
    "n_viajes": conteo,
})
edad_viajes = edad_viajes[(edad_viajes["Edad"] >= 0) & (edad_viajes["Edad"] <= 95)]

print(f"Pearson:  {edad_viajes['Edad'].corr(edad_viajes['n_viajes']):.2f}")
print(f"Spearman: {edad_viajes['Edad'].corr(edad_viajes['n_viajes'], method='spearman'):.2f}")

In [ ]:
curva_edad = edad_viajes.groupby("Edad")["n_viajes"].mean()

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(curva_edad.index, curva_edad.values, color="#4C72B0", linewidth=2)
ax.set_xlabel("Edad (años)")
ax.set_ylabel("Viajes promedio en el día")
ax.set_title("Viajes promedio según edad: la asociación existe, los coeficientes no la ven")
plt.show()

Ambos coeficientes quedan cerca de cero, y sin embargo el gráfico muestra una
asociación clara: los viajes suben hasta los 30 a 40 años y luego caen. La relación
no es monótona, así que el tramo creciente y el decreciente se cancelan: ni Pearson
(que busca una recta) ni Spearman (que busca un orden consistente) están diseñados
para esta forma.

La conclusión que hay que retener: **un coeficiente cercano a cero no demuestra
ausencia de asociación**. Demuestra ausencia de asociación *lineal* (Pearson) o
*monotónica* (Spearman); la forma completa solo se ve en el gráfico.

### Correlación y causalidad

La segunda lección es una distinción que conviene fijar desde ahora. Una
**correlación** describe que dos variables varían juntas. Una **relación causal**
afirma algo más fuerte: que intervenir una variable cambiaría la otra. Los datos
observacionales, como los de esta encuesta, muestran lo primero; no bastan para
establecer lo segundo.

El ejemplo lo ilustra bien. Supongamos que la correlación entre ingreso y uso de
transporte público fuera alta: seguiría sin distinguir entre varias explicaciones
posibles.

- El ingreso reduce el uso de transporte público (más acceso al auto).
- Una tercera variable explica ambas cosas: la **cobertura de la red** afecta el
  uso, y no es independiente de dónde viven los distintos grupos de ingreso. A esta
  tercera variable se le llama **variable de confusión**.
- Alguna combinación de mecanismos operando a la vez.

Distinguir entre estas explicaciones es el problema de la **inferencia causal**, un
área con métodos propios que este curso no cubre. Lo que sí corresponde a este curso
es el hábito defensivo: cuando un análisis muestra una asociación, reportarla como
asociación, y no deslizar un verbo causal ("aumenta", "reduce", "provoca") que los
datos no respaldan.

Y otra vez la pregunta de la clase, ahora a nivel de comunas: cada punto es un
promedio comunal y esconde la variación interna de la comuna. Para el retrato de la
Tarea 1 les corresponde decidir qué números de su comuna reportar y cómo.

## 9. ¿Cambian las rutinas durante la semana?

Cada hogar tiene asignado un día (`DiaAsig` en la tabla de hogares). Con el propósito
del viaje y el factor del día correspondiente podemos comparar las rutinas de lunes a
domingo. Un **heatmap** muestra la tabla completa de una vez.

In [ ]:
proposito = pd.read_csv(RUTA + "tablas_parametros/Proposito.csv", sep=";")

rutinas = (
    viajes.merge(hogares[["Hogar", "DiaAsig"]], on="Hogar", how="left")
    .merge(proposito, left_on="Proposito", right_on="Id", how="left",
           suffixes=("_codigo", ""))
)

# Cada día usa su propia expansión: laboral, sábado o domingo (temporada normal).
rutinas["peso"] = (
    rutinas["FactorLaboralNormal"]
    .fillna(rutinas["FactorSabadoNormal"])
    .fillna(rutinas["FactorDomingoNormal"])
)
rutinas = rutinas.dropna(subset=["peso", "Proposito"])

# "Al trabajo" y "Por trabajo" se agrupan; lo mismo con el estudio.
rutinas["Proposito"] = rutinas["Proposito"].replace({
    "Al trabajo": "Trabajo", "Por trabajo": "Trabajo",
    "Al estudio": "Estudio", "Por estudio": "Estudio",
    "volver a casa": "Volver a casa",
    "Buscar o Dejar a alguien": "Buscar o dejar a alguien",
    "Comer o Tomar algo": "Comer o tomar algo",
    "Otra actividad (especifique)": "Otra actividad",
})

dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
tabla_rutinas = rutinas.pivot_table(
    index="Proposito", columns="DiaAsig", values="peso", aggfunc="sum", fill_value=0,
)[dias]

# Proporción de los viajes de cada día (columnas suman 1)
tabla_rutinas = tabla_rutinas / tabla_rutinas.sum()

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.heatmap(tabla_rutinas, cmap="Blues", annot=True, fmt=".2f",
            cbar_kws={"label": "proporción de los viajes del día"}, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("En qué se viaja cada día de la semana")
plt.show()

Lectura del heatmap: el regreso a casa es cerca de la mitad de los viajes todos los
días (casi todo viaje tiene su vuelta). De lunes a viernes dominan trabajo y estudio;
el fin de semana suben compras, recreación y visitas. La ventaja del heatmap es que
esa comparación completa (14 propósitos por 7 días) cabe en una sola figura legible.

## 10. Una nota sobre el tamaño: Pandas no es para todo

Pandas carga todo en la memoria del computador. Para la EOD (47 MB) sobra; para los
microdatos del Censo 2017 (17,5 millones de personas) o registros de decenas de
gigabytes, se queda sin memoria. Existen herramientas con la misma lógica de tablas
para esos tamaños: **Polars** (sintaxis similar a Pandas, en paralelo), **DuckDB**
(SQL sobre archivos, sin cargar todo), **Dask** y **PySpark** (datos distribuidos).
Las ideas de esta clase (llaves, merge, groupby, factores) son las mismas en todas;
más adelante en el curso las veremos con más detalle.

## 11. Síntesis: ¿de quién habla este número?

Repasamos las herramientas (merge, groupby, describe), pero el aprendizaje de hoy es
la pregunta. Aplíquenla a lo que hicimos:

- *"El hogar promedio gana $688 mil"*: la mayoría de los hogares gana menos; la
  mediana y los percentiles describen mejor al hogar típico.
- *"El 53% de los encuestados son mujeres"*: eso habla de la **muestra**. Para hablar
  de la **ciudad** se suman factores de expansión.
- *"Las personas hacen 1,9 viajes en promedio"*: solo si contamos a las que no
  viajaron. Un `merge` con `how="inner"` las habría excluido del cálculo.
- *"En Vitacura casi no se usa transporte público"*: es un promedio comunal, y
  esconde la variación interna de la comuna.

Las operaciones eran repaso; lo nuevo es examinar qué representa cada cifra.

## Para practicar

1. ¿Qué proporción de los encuestados tiene licencia de conducir? ¿Cambia si la
   calcula ponderada por factor de expansión?
2. Calcule la edad mediana por comuna del hogar y encuentre las cinco comunas más
   jóvenes. Necesita un merge entre `personas` y `hogares`.
3. ¿Cuál es el propósito de viaje más frecuente después del regreso a casa? Compare
   el resultado con y sin factores de expansión.
4. Los viajes en transporte público, ¿duran más o menos que los viajes en auto?
   Compare medianas y explique por qué no compara medias.

## Referencias y créditos

- SECTRA, Ministerio de Transportes (2014). *Encuesta Origen Destino de Viajes
  Santiago 2012*. https://www.sectra.gob.cl/biblioteca/detalle1.asp?mfn=3253
- El enfoque de este notebook (preguntas sobre la ciudad con la EOD y el uso de los
  factores de expansión) está inspirado en el material del curso de visualización de
  Eduardo Graells-Garrido: https://github.com/zorzalerrante/aves
- Bruce, P., Bruce, A. y Gedeck, P. (2020). *Practical Statistics for Data
  Scientists*, 2ª ed. O'Reilly. Capítulo 1.
- Chambers, J. M. y Hastie, T. J. (1992). *Statistical Models in S*. Wadsworth &
  Brooks/Cole.
- McKinney, W. (2010). *Data Structures for Statistical Computing in Python*.
  Proceedings of the 9th Python in Science Conference.